Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Webhooks

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Webhooks.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

Preview: Webhooks support in the Gemini API is currently in preview.

Webhooks allow the Gemini API to push real-time notifications to your server
when asynchronous or Long-Running Operations (LROs) complete. This replaces the
need to poll the API for status updates, reducing latency and overhead.

Webhooks are available for operations like [Batch](https://ai.google.dev/gemini-api/docs/batch) jobs,
[Interactions](https://ai.google.dev/gemini-api/docs/interactions) and [video generation](https://ai.google.dev/gemini-api/docs/video).

## How it works

Instead of polling `GET /operations` repeatedly to check if a job is finished,
you can configure Gemini API Webhooks to send an HTTP POST request to your
listener URL immediately upon an event trigger.

The Gemini API supports two ways to configure webhooks:

* **Static webhooks**: Project-level endpoints. Good for global integrations (e.g., notifying Slack, syncing a
database, etc.).
* **Dynamic webhooks**: Request-level overrides passing a
webhook URL in the configuration payload of a specific jobs call. Ideal for
routing specific jobs to dedicated endpoints.


# Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai).

In [ ]:
%pip install -U -q "google-genai>=1.73.1" # Minimum version 1.73.1 for webhooks

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

Initialize a client with your API key:

In [ ]:
import os
from google import genai
from google.genai import errors, types

client = genai.Client(api_key=GEMINI_API_KEY)

## Step 1: Create a webhook endpoint

This example uses Cloudflare workers, which allow you to easily deploy a serverless function you can use as our webhook endpoint.

1. Go to https://developers.cloudflare.com/workers/examples/read-post/
2. Click deploy to Cloudflare and finish deployment
3. Cloudlfare will create a GitHub repository for you and you'll get a URL you can use as webhook

(Their management UI at https://dash.cloudflare.com/ has a log observation page, you can see the requests there)

4. **Implement webhook signature verification**: In your repository, update the `package.json` dependencies with

```jsdo
"dependencies": {
		"standardwebhooks": "^1.0.0"
	}
```


 and replace the code in `/src/index.ts` with the following code. A final worker implementation can be found here: https://github.com/patrickloeber/gemini-api-webhook-example.

```ts
import { Webhook } from "standardwebhooks";

export default {
	async fetch(request, env): Promise<Response> {
		// Handle POST requests
		if (request.method === "POST") {
			const payload = await request.text();
			const headers: Record<string, string> = {};
			request.headers.forEach((value, key) => {
				headers[key] = value;
			});

			try {
				const wh = new Webhook(env.WEBHOOK_SIGNING_SECRET);
				const event = wh.verify(payload, headers) as Record<string, any>;

				// Process thin payload contents
				if (event.type === "batch.completed" || event.type === "video.generated") {
					const uri = event.data.output_file_uri;
					console.log(`Job finished! Results at: ${uri}`);
				}

				return Response.json({ status: "received" }, { status: 200 });
			} catch (e) {
				console.error("Webhook verification failed:", e);
				return Response.json({ error: "Signature invalid" }, { status: 400 });
			}
		}

		if (request.method === "GET") {
			return new Response("Worker is running!");
		}

		return new Response("Not Found", { status: 404 });
	},
} satisfies ExportedHandler<Env>;
```

 When an event happens that you're subscribed to, your webhook URL will receive an HTTP POST request. Your endpoint must respond with a 2xx status code within a few seconds to avoid a retry. To ensure delivery, the Gemini API automatically retries failed requests for 24 hours using exponential backoff.


TODO: Change this to use your own webhook enpoint!

In [ ]:
# TODO: Change this to use your own webhook enpoint, must be https
WEBHOOK_ENDPOINT="https://read-post.pythonengineerman.workers.dev/"  # @param {type:"string"}

Check that the endpoint is working. This example endpoint simply returns a text for a GET request.

In [ ]:
import requests

r = requests.get(WEBHOOK_ENDPOINT)
print(r.text)

Worker is running!


## Step 2: Create a webhook

**IMPORTANT**: When creating a webhook, the API returns a **signing secret**
**only once**. You must store this securely to verify signatures later.

In [ ]:
webhook = client.webhooks.create(
    subscribed_events=["video.generated"],
    name="video_generation_webhook",
    uri=WEBHOOK_ENDPOINT,
)

# Store webhook.signing_secret securely (e.g., in your env vars)
os.environ["WEBHOOK_SIGNING_SECRET"] = webhook.new_signing_secret

# Or write it to a file:
with open(".env", "a") as f:
    f.write(f"WEBHOOK_SIGNING_SECRET={webhook.new_signing_secret}\n")

print(f"Created webhook: {webhook.name}")


Created webhook: video_generation_webhook


### List

List all configured webhooks for the current project, with optional pagination.

In [ ]:
client.webhooks.list()

WebhookListResponse(next_page_token=None, webhooks=[Webhook(subscribed_events=['video.generated'], uri='https://read-post.pythonengineerman.workers.dev/', id='k0h9q38f6ejj1g9j7cgcdsauadmbewegl79f', create_time=None, name='video_generation_webhook', new_signing_secret=None, signing_secrets=[SigningSecret(expire_time=None, truncated_secret='whsec_...3frh')], state='enabled', update_time=None)])

### Get/Ping/Delete/Update

Ping a webhook

In [ ]:
client.webhooks.ping(id=webhook.id)

WebhookPingResponse()

Get a webhook

In [ ]:
wh =client.webhooks.get(id=webhook.id)
wh

Webhook(subscribed_events=['video.generated'], uri='https://read-post.pythonengineerman.workers.dev/', id='k0h9q38f6ejj1g9j7cgcdsauadmbewegl79f', create_time=None, name='video_generation_webhook', new_signing_secret=None, signing_secrets=[SigningSecret(expire_time=None, truncated_secret='whsec_...3frh')], state='enabled', update_time=None)

This would delete the webhook. Set `delete=True` to run it:

In [ ]:
delete = False
if delete:
    client.webhooks.delete(id=webhook.id)

This is how you can update a webhook. Set `update=True` to run it:

In [ ]:
update = False
if update:
    wh = client.webhooks.update(
        id=webhook.id,
        subscribed_events=["video.generated", "batch.succeeded"],
        name="video_and_batch_webhook",
    )

## Step 3: Update SECRET in webhook endpoint

Store the `WEBHOOK_SIGNING_SECRET` you stored above as an environment variable for your Cloudflare worker.

This can be done either in the UI or via your terminal. For more info, see https://developers.cloudflare.com/workers/configuration/secrets/

```bash
# Set the secret (one-time, you'll be prompted to paste the value)
npx wrangler secret put WEBHOOK_SIGNING_SECRET
# Deploy
npm run deploy
```

## Step 4: Send a Gemini API request

Senf a request for the long running operation your webhook is subscribed to.

In [ ]:
prompt = """A stunning drone view of the Grand Canyon during a flamboyant sunset that highlights the canyon's colors. The drone slowly flies towards the sun then accelerates, dives and flies inside the canyon."""

operation = client.models.generate_videos(
    model="veo-3.1-generate-preview",
    prompt=prompt,
)

Now, instead of polling GET /operations, your webhook should receive an HTTP POST request once the video generation has finished.

## Step 5: Check the worker logs

Check the **Observability** tab in your Cloudflare worker dashboard (https://dash.cloudflare.com/).

It should have a log statement that outputs the `output_file_uri`. You can use this URI to access the generated data.

## Dynamic webhook config

You can also use dynamic webhook configuration, but for this, you need to update your webhook verfification.

See the docs for implementation details:

- https://ai.google.dev/gemini-api/docs/webhooks#dynamic-webhooks
- https://ai.google.dev/gemini-api/docs/webhooks#verify-dynamic-signatures


## Next Steps:

- **Webhooks documentation**: Read the full [Webhooks guide](https://ai.google.dev/gemini-api/docs/webhooks) for advanced configuration options, supported event types, and best practices.
- **Batch API**: Webhooks pair well with the [Batch API](./Batch_mode.ipynb) to process large volumes of requests and get notified when they complete.
- **Video generation**: Use webhooks with [Veo](./Get_started_Veo.ipynb) to get notified when your generated videos are ready.
- **Cloudflare Workers**: Learn more about building webhook endpoints with [Cloudflare Workers documentation](https://developers.cloudflare.com/workers/).
- **Standard Webhooks**: The Gemini API uses the [Standard Webhooks](https://www.standardwebhooks.com/) specification for signature verification, ensuring secure and interoperable webhook delivery.
